In [1]:
!pip install word2number
!pip install joblib

  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
  Created wheel for word2number: filename=word2number-1.1-py3-none-any.whl size=5588 sha256=2a47fcf0305682054acb81ee836ee6998972211b765f44fd1e4a7943cac21763
  Stored in directory: c:\users\hp\appdata\local\pip\cache\wheels\cd\ef\ae\073b491b14d25e2efafcffca9e16b2ee6d114ec5c643ba4f06
Successfully built word2number


In [2]:
import numpy as np
import pandas as pd
from word2number import w2n
from sklearn import linear_model
import math
import pickle
import joblib

In [3]:
# Load the dataset
df = pd.read_csv('data/hiring.csv')

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8 entries, 0 to 7
Data columns (total 4 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   experience                  6 non-null      object 
 1   test_score(out of 10)       7 non-null      float64
 2   interview_score(out of 10)  8 non-null      int64  
 3   salary($)                   8 non-null      int64  
dtypes: float64(1), int64(2), object(1)
memory usage: 388.0+ bytes


In [4]:
df

,experience,test_score(out of 10),interview_score(out of 10),salary($)
0,NaN,8.0,9,50000
1,NaN,8.0,6,45000
2,five,6.0,7,60000
3,two,10.0,10,65000
4,seven,9.0,6,70000
5,three,7.0,10,62000
6,ten,NaN,7,72000
7,eleven,7.0,8,80000


In [5]:
df['experience'] = df['experience'].apply(lambda x: w2n.word_to_num(x) if isinstance(x, str) else None)
df.head()

,experience,test_score(out of 10),interview_score(out of 10),salary($)
0,NaN,8.0,9,50000
1,NaN,8.0,6,45000
2,5.0,6.0,7,60000
3,2.0,10.0,10,65000
4,7.0,9.0,6,70000


In [6]:
df.experience = df.experience.fillna(0)
df.head()

,experience,test_score(out of 10),interview_score(out of 10),salary($)
0,0.0,8.0,9,50000
1,0.0,8.0,6,45000
2,5.0,6.0,7,60000
3,2.0,10.0,10,65000
4,7.0,9.0,6,70000


In [7]:
median_test = df['test_score(out of 10)'].median()
df['test_score(out of 10)'].fillna(median_test, inplace=True)
df

,experience,test_score(out of 10),interview_score(out of 10),salary($)
0,0.0,8.0,9,50000
1,0.0,8.0,6,45000
2,5.0,6.0,7,60000
3,2.0,10.0,10,65000
4,7.0,9.0,6,70000
5,3.0,7.0,10,62000
6,10.0,8.0,7,72000
7,11.0,7.0,8,80000


In [8]:
lin_reg = linear_model.LinearRegression()
lin_reg.fit(df[['experience', 'test_score(out of 10)', 'interview_score(out of 10)']], df['salary($)'])

LinearRegression()

In [9]:
lin_reg.predict([[2, 9, 6]])

c:\Users\HP\anaconda3\Lib\site-packages\sklearn\base.py:439: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(


array([53205.96797671])

In [10]:
lin_reg.predict([[12, 10, 10]])

c:\Users\HP\anaconda3\Lib\site-packages\sklearn\base.py:439: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(


array([92002.18340611])

## salary = a*experience + b*test_score + c*interview_score + d

- salary -> prediction
- experience, test_score, interview_score -> independent variables
- a, b, c -> coef
- d -> intercept

In [11]:
lin_reg.coef_

array([2812.95487627, 1845.70596798, 2205.24017467])

In [12]:
lin_reg.intercept_

17737.26346433771

In [13]:
# y = a1*x1 + a2*x2 + a3*x3 + d
2*lin_reg.coef_[0] + 9*lin_reg.coef_[1] + 6*lin_reg.coef_[2] + lin_reg.intercept_

53205.96797671034

# Saving The Model

1. using pickel
2. using joblit

In [14]:
with open('model.pkl', 'wb') as file:
    pickle.dump(lin_reg, file)

In [15]:
with open('model.pkl', 'rb') as file:
    mp = pickle.load(file)

In [16]:
mp.predict([[2, 9, 6]])

c:\Users\HP\anaconda3\Lib\site-packages\sklearn\base.py:439: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(


array([53205.96797671])

In [17]:
mp.coef_

array([2812.95487627, 1845.70596798, 2205.24017467])

In [18]:
mp.intercept_

17737.26346433771

In [19]:
joblib.dump(lin_reg, 'model.joblib')

['model.joblib']

In [20]:
mj = joblib.load('model.joblib')

In [21]:
mj.predict([[2, 9, 6]])

c:\Users\HP\anaconda3\Lib\site-packages\sklearn\base.py:439: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(


array([53205.96797671])

In [22]:
mj.coef_

array([2812.95487627, 1845.70596798, 2205.24017467])

In [23]:
mj.intercept_

17737.26346433771